In [2]:
import scipy.io as sio
import yaml
import numpy as np

# 加载 .mat 文件
mat_data = sio.loadmat('stanford_cars/devkit/cars_meta.mat')  # 替换为实际文件名
class_names_raw = mat_data['class_names']

# 提取类别名称（确保为普通字符串）
class_names = [str(name[0]) for name in class_names_raw.flatten()]

# 构建 data.yaml 内容
data_dict = {
    'path': '/path/to/dataset',  
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names),
    'names': class_names
}

# 写入文件，注意编码和 allow_unicode
with open('data.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(data_dict, f, 
              default_flow_style=False, 
              allow_unicode=True,    
              sort_keys=False)        
print("data.yaml 已生成，共 {} 个类别".format(len(class_names)))

data.yaml 已生成，共 196 个类别


In [ ]:
import scipy.io
import cv2
import os
import numpy as np

# 路径设置（根据实际下载位置修改）
train_annos_path = 'stanford_cars/devkit/cars_train_annos.mat'
test_annos_path = 'stanford_cars/devkit/cars_test_annos_withlabels.mat'
meta_path = 'stanford_cars/devkit/cars_meta.mat'

train_img_dir = 'images/train/'
val_img_dir = 'images/test/'
train_label_dir = 'labels/train/'
val_label_dir = 'labels/val/'

# 创建输出目录
os.makedirs(train_label_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# 1. 读取类别名称
meta = scipy.io.loadmat(meta_path)
class_names = [name[0] for name in meta['class_names'][0]]  # 196 个类别名称

# 2. 处理训练集
train_data = scipy.io.loadmat(train_annos_path)
annotations = train_data['annotations'][0]  # 结构体数组

for ann in annotations:
    # 提取字段（注意每个字段都是 shape (1,1) 的 numpy 数组，需要 [0][0] 取值）
    bbox_x1 = int(ann['bbox_x1'][0][0])
    bbox_y1 = int(ann['bbox_y1'][0][0])
    bbox_x2 = int(ann['bbox_x2'][0][0])
    bbox_y2 = int(ann['bbox_y2'][0][0])
    class_id = int(ann['class'][0][0]) - 1   # 类别从 1 开始，转为 0-based
    fname = str(ann['fname'][0])

    # 图片路径
    img_path = os.path.join(train_img_dir, fname)
    if not os.path.exists(img_path):
        print(f"Warning: {img_path} not found, skipping.")
        continue

    # 获取图片尺寸
    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # 转换为 YOLO 归一化坐标
    x_center = (bbox_x1 + bbox_x2) / 2.0 / w
    y_center = (bbox_y1 + bbox_y2) / 2.0 / h
    width = (bbox_x2 - bbox_x1) / w
    height = (bbox_y2 - bbox_y1) / h

    # 写入 .txt 文件（与图片同名）
    txt_path = os.path.join(train_label_dir, fname.replace('.jpg', '.txt'))
    with open(txt_path, 'w') as f:
        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

print(f"训练集转换完成，共处理 {len(annotations)} 张图片。")

# 3. 处理测试集（作为验证集）
test_data = scipy.io.loadmat(test_annos_path)
annotations = test_data['annotations'][0]

for ann in annotations:
    bbox_x1 = int(ann['bbox_x1'][0][0])
    bbox_y1 = int(ann['bbox_y1'][0][0])
    bbox_x2 = int(ann['bbox_x2'][0][0])
    bbox_y2 = int(ann['bbox_y2'][0][0])
    class_id = int(ann['class'][0][0]) - 1
    fname = str(ann['fname'][0])

    img_path = os.path.join(val_img_dir, fname)
    if not os.path.exists(img_path):
        print(f"Warning: {img_path} not found, skipping.")
        continue

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    x_center = (bbox_x1 + bbox_x2) / 2.0 / w
    y_center = (bbox_y1 + bbox_y2) / 2.0 / h
    width = (bbox_x2 - bbox_x1) / w
    height = (bbox_y2 - bbox_y1) / h

    txt_path = os.path.join(val_label_dir, fname.replace('.jpg', '.txt'))
    with open(txt_path, 'w') as f:
        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

print(f"验证集转换完成，共处理 {len(annotations)} 张图片。")